In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "6"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
from tqdm import tqdm
from alignment.prompts import OEQA_INSTRUCTION_TEMPLATE, OEQA_SYSTEM_PROMPT, MCQA_SYSTEM_PROMPT, MCQA_INSTRUCTION_TEMPLATE

tokenizer = AutoTokenizer.from_pretrained("/gpfs/home/eungizoa/models/doctrio/qwen3-14b-mcqa-oeqa")
model = AutoModelForCausalLM.from_pretrained("/gpfs/home/eungizoa/models/doctrio/qwen3-14b-mcqa-oeqa", torch_dtype=torch.bfloat16).to("cuda:0")

raw_datasets = pd.read_csv("/gpfs/home/eungizoa/data/hydra-fineval/preliminary/test.parsed.csv").to_dict(orient="records")
for example in raw_datasets:
    example["options"] = eval(example["options"])
    if example["options"]:
        system_message = MCQA_SYSTEM_PROMPT
        options_str = "\n".join([f"{i+1}. {option}" for i, option in enumerate(example["options"])])
        user_message = MCQA_INSTRUCTION_TEMPLATE.format(query=example["query"], options=options_str)
    else:
        system_message = OEQA_SYSTEM_PROMPT
        user_message = OEQA_INSTRUCTION_TEMPLATE.format(query=example["query"])
        
    example["messages"] = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [2]:
model.eval()

with torch.no_grad():
    
    for example in tqdm(raw_datasets):
        model_inputs = tokenizer.apply_chat_template(example["messages"], tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors="pt", return_dict=True)
        outputs = model.generate(**model_inputs.to(model.device), max_new_tokens=2048)
        response_ids = outputs[0][len(model_inputs["input_ids"][0]):].tolist()
        response = tokenizer.decode(response_ids, skip_special_tokens=True)
        example["response"] = response

100%|██████████| 515/515 [1:28:40<00:00, 10.33s/it]


In [6]:
import os
import re

def extract_cot_and_final_answer(text):
    """
    Extract thinking traces and final answer from text.
    
    Returns:
        tuple: (thinking_trace, answer)
    """
    # Pattern to match thinking traces (everything before "Answer:")
    thinking_pattern = r'^(.*?)(?=Answer:\s*)'
    
    # Pattern to match the answer after "Answer:"
    answer_pattern = r'.*Answer:\s*(.*)$'
    
    thinking_match = re.search(thinking_pattern, text, re.DOTALL | re.MULTILINE)
    answer_match = list(re.finditer(answer_pattern, text, re.DOTALL | re.MULTILINE))
    if answer_match:
        answer_match = answer_match[-1]
    else:
        return "", ""
    
    thinking_trace = thinking_match.group(1).strip() if thinking_match else ""
    answer = answer_match.group(1).strip() if answer_match else ""
    
    return thinking_trace.strip(), answer.strip()

for example in raw_datasets:
    cot_trace, answer = extract_cot_and_final_answer(example["response"])
    example["is_mcqa"] = True if example["options"] else False
    if example["is_mcqa"]:
        example["parsed_cot_trace"] = cot_trace
        example["parsed_answer"] = answer if answer else "0"
    else:
        example["parsed_cot_trace"] = cot_trace
        example["parsed_answer"] = answer

In [7]:
submission = pd.DataFrame([{"ID": example["ID"], "Answer": example["parsed_answer"], "is_mcqa": example["is_mcqa"]} for example in raw_datasets])

In [8]:
submission[submission["is_mcqa"] == True]["Answer"].value_counts()

Answer
3    122
4    122
1    109
2     96
5     47
0      4
Name: count, dtype: int64

In [9]:
submission[(submission["is_mcqa"] == True) & (submission["Answer"] == "0")]

,ID,Answer,is_mcqa
169,TEST_169,0,True
179,TEST_179,0,True
209,TEST_209,0,True
281,TEST_281,0,True


In [11]:
[example for example in raw_datasets if example["ID"] == "TEST_169"][0]

{'Unnamed: 0': 169,
 'ID': 'TEST_169',
 'Question': '네트워크 공유의 동작 원리와 관련된 프로토콜이 아닌 것은?\n1 Netbios\n2 HTTPS\n3 SMTP\n4 Netbeui\n5 P2P',
 'query': '네트워크 공유의 동작 원리와 관련된 프로토콜이 아닌 것은?',
 'options': ['Netbios', 'HTTPS', 'SMTP', 'Netbeui', 'P2P'],
 'messages': [{'role': 'system',
   'content': 'You are a knowledgeable assistant that answers multiple-choice questions. Choose the correct option based on the question and options provided. Explain your reasoning if possible.'},
  {'role': 'user',
   'content': 'Question: 네트워크 공유의 동작 원리와 관련된 프로토콜이 아닌 것은?\nOptions: \n1. Netbios\n2. HTTPS\n3. SMTP\n4. Netbeui\n5. P2P\n\nProvide a detailed reasoning to solve the question if possible. End your response with the correct option in the format "Answer: $OPTION".'}],
 'response': '주어진 파일 내용에 따르면, 네트워크 공유의 동작 원리와 관련된 프로토콜로는 Netbios, NetBEUI, P2P, IPX/SPX가 언급되어 있습니다. 이 중에서 문제에 제시된 선택지 중 "HTTPS"와 "SMTP"는 파일에 전혀 언급되지 않았으며, 네트워크 공유와 직접적인 관련이 없는 프로토콜입니다. 그러나 문제는 "네트워크 공유의 동작 원리와 관련된 프로토콜이 아닌 것"을 묻고 있으며, 주어진 파일 내용

In [12]:
submission.drop(columns=["is_mcqa"]).to_csv("/gpfs/home/eungizoa/data/hydra-fineval/submission/submission-250825-13pm.csv", encoding="utf-8-sig", index=None)